# Paper Figures & Tables

Generates all publication-quality figures and tables for the WheatGPCPipeline paper.

**Figures:**
1. Study Area Map
2. Methodological Framework (external - Figma/draw.io)
3. Phenological Peak Detection
4. Temporal Strategies Schematic
5. Performance Heatmap (all combinations)
6. Temporal Zoom-In
7. Predicted vs Observed
8. SHAP Beeswarm

**Tables:**
1. Main Results (2TN Γ— 3TA Γ— 3M)
2. Top Selected Features

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

# Paths
ROOT = Path(r"../data")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
NB_PROC = Path(fr'../notebooks/data/processed')
MODELS = Path(fr'../models')
TEMPORAL = Path(fr'../models/temporal_combinations')
FIGURES_OUT = ROOT / "paper" / "figures"
FIGURES_OUT.mkdir(parents=True, exist_ok=True)

# Plotly template for publication
pio.templates.default = "plotly_white"

# Common styling
FONT_FAMILY = "Arial"
FONT_SIZE = 14
AXIS_FONT = dict(family=FONT_FAMILY, size=FONT_SIZE)
TITLE_FONT = dict(family=FONT_FAMILY, size=16, color="black")

# Color palette (colorblind-friendly)
MODEL_COLORS = {
    "RandomForest": "#2196F3",
    "LightGBM": "#4CAF50",
    "XGBoost": "#FF9800",
    "Ensemble": "#9C27B0",
    "ElasticNet": "#795548",
    "Stacking": "#607D8B",
}

NORM_COLORS = {
    "Peak-relative": "#E63946",
    "Calendar": "#457B9D",
}

print("Setup complete.")

## Load All Data

In [ ]:
# --- All fields with protein data (240 total, 228 after QC) ---
static_full = pd.read_csv(DATA_PROC / "static_features_full.csv")
print(f"Total fields (static_features_full): {len(static_full)}")
print(f"Protein range: {static_full['protein_pct'].min():.2f} - {static_full['protein_pct'].max():.2f}%")
print(f"Protein mean: {static_full['protein_pct'].mean():.2f} Β± {static_full['protein_pct'].std():.2f}%")

# --- Peak detection results ---
peak_days = pd.read_csv(NB_PROC / "peak_days.csv")
dl_params = pd.read_csv(NB_PROC / "double_logistic_params.csv")
with open(NB_PROC / "peak_qc_report.json") as f:
    peak_qc = json.load(f)
print(f"\nPeak QC: {peak_qc['valid_fields']}/{peak_qc['total_fields']} valid fields")
print(f"Peak DOY: {peak_qc['peak_doy_stats']['mean']:.1f} Β± {peak_qc['peak_doy_stats']['std']:.1f}")

# --- Merge: keep only 228 QC-valid fields ---
valid_peaks = peak_days[~peak_days["peak_anomalous"]]
fields_228 = static_full.merge(
    valid_peaks[["field_key", "peak_doy", "peak_gcvi", "fit_r2"]],
    on="field_key",
    how="inner",
)
print(f"\nValid fields (228 after QC merge): {len(fields_228)}")

# --- Model results (all strategies) ---
results = {}
for f in sorted(MODELS.glob("results_*.json")):
    strategy = f.stem.replace("results_", "")
    with open(f) as fh:
        results[strategy] = json.load(fh)
print(f"\nStrategies loaded: {list(results.keys())}")

# --- OOF predictions ---
oof = {}
for f in sorted(MODELS.glob("oof_predictions_*.csv")):
    strategy = f.stem.replace("oof_predictions_", "")
    oof[strategy] = pd.read_csv(f)
print(f"OOF predictions loaded: {list(oof.keys())}")

# --- Temporal combination results ---
combo_results = {}
for f in sorted(TEMPORAL.glob("results_all_*_v3.csv")):
    name = f.stem.replace("results_all_", "").replace("_v3", "")
    combo_results[name] = pd.read_csv(f)
print(f"\nTemporal combo results: {list(combo_results.keys())}")

---
## Figure 1: Study Area Map

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle, FancyArrow
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
import geopandas as gpd
from shapely.geometry import Point

# ============================================================
# Data
# ============================================================
geometry = [Point(xy) for xy in zip(fields_228["centroid_lon"], fields_228["centroid_lat"])]
gdf = gpd.GeoDataFrame(fields_228, geometry=geometry, crs="EPSG:4326")

protein = gdf["protein_pct"].values
lon_arr = gdf["centroid_lon"].values
lat_arr = gdf["centroid_lat"].values

# Extent with padding
pad_lon, pad_lat = 0.45, 0.35
extent = [lon_arr.min() - pad_lon, lon_arr.max() + pad_lon,
          lat_arr.min() - pad_lat, lat_arr.max() + pad_lat]

proj = ccrs.PlateCarree()

# ============================================================
# Figure
# ============================================================
fig = plt.figure(figsize=(7.5, 6.5), dpi=250)
ax = fig.add_axes([0.06, 0.06, 0.76, 0.88], projection=proj)
ax.set_extent(extent, crs=proj)

# ---- Background: states ----
states_shp = shpreader.natural_earth(
    resolution="50m", category="cultural", name="admin_1_states_provinces"
)
for rec in shpreader.Reader(states_shp).records():
    name = rec.attributes.get("name", "")
    if name in ("Kansas", "Nebraska", "Colorado", "Oklahoma", "Missouri"):
        fc = "white" if name == "Kansas" else "#f5f5f5"
        ec = "#333333" if name == "Kansas" else "#666666"
        lw = 0.9 if name == "Kansas" else 0.5
        ax.add_geometries([rec.geometry], proj, facecolor=fc, edgecolor=ec, linewidth=lw, zorder=1)

# ---- County boundaries (Kansas) ----
counties_shp = shpreader.natural_earth(
    resolution="10m", category="cultural", name="admin_2_counties"
)
for rec in shpreader.Reader(counties_shp).records():
    if rec.attributes.get("STATEFP") == "20":
        ax.add_geometries(
            [rec.geometry], proj,
            facecolor="none", edgecolor="#d0d0d0", linewidth=0.2, zorder=2,
        )

# ---- Hydrography ----
ax.add_feature(cfeature.RIVERS.with_scale("50m"), edgecolor="#b0d4e8", linewidth=0.4, zorder=2)
ax.add_feature(cfeature.LAKES.with_scale("50m"), facecolor="#daeaf5", edgecolor="#b0d4e8", linewidth=0.2, zorder=2)

# ---- Field points ----
vmin, vmax = np.percentile(protein, [2, 98])
sc = ax.scatter(
    lon_arr, lat_arr,
    c=protein, cmap="RdYlGn",
    s=22, edgecolors="black", linewidths=0.3,
    vmin=vmin, vmax=vmax, alpha=0.92, zorder=5,
    transform=proj,
)

# ---- Colorbar ----
cax = fig.add_axes([0.84, 0.18, 0.022, 0.50])
cbar = fig.colorbar(sc, cax=cax, extend="both")
cbar.set_label("Grain Protein Content (%)", fontsize=9, fontfamily="Arial", labelpad=8)
cbar.ax.tick_params(labelsize=8)
cbar.outline.set_linewidth(0.4)

# ---- Lat/Lon grid ----
gl = ax.gridlines(
    draw_labels=True, linewidth=0.4, color="#888888", alpha=0.5,
    linestyle="--", x_inline=False, y_inline=False,
)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {"size": 8, "fontfamily": "Arial"}
gl.ylabel_style = {"size": 8, "fontfamily": "Arial"}
gl.xlocator = mticker.MultipleLocator(1)
gl.ylocator = mticker.MultipleLocator(0.5)

# ---- State abbreviations ----
for abbr, lx, ly in [("KS", -98.6, 38.05), ("NE", -99.8, 39.55),
                      ("CO", -102.35, 38.5), ("OK", -99.8, 37.12)]:
    if extent[0] < lx < extent[1] and extent[2] < ly < extent[3]:
        ax.text(lx, ly, abbr, fontsize=8, color="#aaaaaa", fontstyle="italic",
                fontfamily="Arial", ha="center", va="center", transform=proj, zorder=3)

# ---- Reference cities ----
cities = [
    ("Hays", -99.327, 38.879),
    ("Dodge City", -100.017, 37.753),
    ("Colby", -101.052, 39.396),
    ("Great Bend", -98.765, 38.364),
]
for cname, cx, cy in cities:
    if extent[0] + 0.2 < cx < extent[1] - 0.2 and extent[2] + 0.15 < cy < extent[3] - 0.15:
        ax.plot(cx, cy, marker="s", color="black", markersize=3, transform=proj, zorder=6)
        ax.text(cx + 0.08, cy + 0.06, cname, fontsize=6, fontfamily="Arial",
                color="#444444", transform=proj, zorder=6)

# ---- Scale bar (bottom-right) ----
sb_x0 = extent[1] - 0.35
sb_y0 = extent[2] + 0.10
bar_km = 50
bar_deg = bar_km / (111.32 * np.cos(np.radians(sb_y0)))

ax.plot([sb_x0 - bar_deg, sb_x0], [sb_y0, sb_y0],
        "k-", linewidth=1.8, transform=proj, zorder=10)
for xp in [sb_x0 - bar_deg, sb_x0 - bar_deg / 2, sb_x0]:
    ax.plot([xp, xp], [sb_y0 - 0.025, sb_y0 + 0.025],
            "k-", linewidth=1, transform=proj, zorder=10)

ax.text(sb_x0 - bar_deg, sb_y0 - 0.06, "0", fontsize=6, ha="center",
        fontfamily="Arial", transform=proj, zorder=10)
ax.text(sb_x0, sb_y0 - 0.06, f"{bar_km}", fontsize=6, ha="center",
        fontfamily="Arial", transform=proj, zorder=10)
ax.text(sb_x0 - bar_deg / 2, sb_y0 + 0.06, "km", fontsize=6, ha="center",
        fontfamily="Arial", transform=proj, zorder=10)

# ---- North arrow (top-right) ----
na_x = extent[1] - 0.20
na_y = extent[3] - 0.20
ax.annotate("", xy=(na_x, na_y + 0.18), xytext=(na_x, na_y),
            arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black"),
            transform=proj, zorder=10)
ax.text(na_x, na_y + 0.22, "N", fontsize=8, fontweight="bold", fontfamily="Arial",
        ha="center", va="bottom", transform=proj, zorder=10)

# ---- Sample size annotation ----
stats_text = (
    f"n = {len(fields_228)} fields\n"
    f"Mean GPC = {protein.mean():.1f}% (SD {protein.std():.1f}%)\n"
    f"Range: {protein.min():.1f} - {protein.max():.1f}%"
)
ax.text(0.02, 0.02, stats_text,
        transform=ax.transAxes, fontsize=7, fontfamily="Arial",
        verticalalignment="bottom",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#cccccc", alpha=0.9),
        zorder=10)

# ============================================================
# Inset: contiguous US
# ============================================================
ax_in = fig.add_axes([0.07, 0.60, 0.24, 0.24],
                     projection=ccrs.AlbersEqualArea(central_longitude=-96, central_latitude=37.5))
ax_in.set_extent([-125, -66, 24, 50], crs=proj)

ax_in.add_feature(cfeature.LAND, facecolor="#f2f2f2", edgecolor="none")
ax_in.add_feature(cfeature.OCEAN, facecolor="#e6eff8")
ax_in.add_feature(cfeature.STATES.with_scale("50m"), edgecolor="#aaaaaa", linewidth=0.25)
ax_in.add_feature(cfeature.COASTLINE.with_scale("50m"), edgecolor="#777777", linewidth=0.4)

# Kansas fill
for rec in shpreader.Reader(states_shp).records():
    if rec.attributes.get("name") == "Kansas":
        ax_in.add_geometries([rec.geometry], proj,
                             facecolor="#ffcccc", edgecolor="#cc0000",
                             linewidth=1.0, alpha=0.8)
        break

# Study area box
rect = Rectangle(
    (lon_arr.min() - 0.15, lat_arr.min() - 0.1),
    (lon_arr.max() - lon_arr.min()) + 0.3,
    (lat_arr.max() - lat_arr.min()) + 0.2,
    linewidth=1.5, edgecolor="red", facecolor="none",
    transform=proj, zorder=10,
)
ax_in.add_patch(rect)

for sp in ax_in.spines.values():
    sp.set_edgecolor("black")
    sp.set_linewidth(0.6)

# ============================================================
# Save
# ============================================================
fig.savefig(FIGURES_OUT / "fig1_study_area.png", dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(FIGURES_OUT / "fig1_study_area.pdf", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved fig1_study_area.png and .pdf")

---
## Figure 3: Phenological Peak Detection

In [ ]:
import sys
_REPO_ROOT = Path("..").resolve()  # paper/ -> repo root
sys.path.insert(0, str(_REPO_ROOT / "src"))
from double_logistic import double_logistic

# Load raw spectral data (one month to demo)
spec_files = sorted((DATA_RAW / "spectral" / "GEE_Wheat_Daily_S2_FullSeason").glob("*.csv"))
spectral_dfs = []
for f in spec_files:
    df = pd.read_csv(f)
    spectral_dfs.append(df)
spectral = pd.concat(spectral_dfs, ignore_index=True)
spectral["date"] = pd.to_datetime(spectral["date"])
spectral["doy"] = spectral["date"].dt.dayofyear + (
    (spectral["date"].dt.year - 2024) * 365
)  # continuous DOY across years

print(f"Spectral data: {len(spectral)} rows, {spectral['field_key'].nunique()} fields")
print(f"Date range: {spectral['date'].min()} to {spectral['date'].max()}")

In [ ]:
from scipy.signal import savgol_filter

# Select 4 representative fields (good fits, different peak timing)
valid_params = dl_params.merge(valid_peaks[["field_key", "peak_doy", "peak_gcvi"]], on="field_key")
sorted_by_doy = valid_params.sort_values("peak_doy")
n = len(sorted_by_doy)
example_fields = [
    sorted_by_doy.iloc[int(n * 0.15)]["field_key"],  # early peak
    sorted_by_doy.iloc[int(n * 0.40)]["field_key"],  # mid-early
    sorted_by_doy.iloc[int(n * 0.60)]["field_key"],  # mid-late
    sorted_by_doy.iloc[int(n * 0.85)]["field_key"],  # late peak
]

fig3 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"Field: {fk}" for fk in example_fields],
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

# Display & fit window: Feb 1 β€“ Jun 30, 2025
fit_window_start = pd.Timestamp("2025-02-01")
fit_window_end = pd.Timestamp("2025-06-30")

for idx, fk in enumerate(example_fields):
    row, col = idx // 2 + 1, idx % 2 + 1

    # --- Raw GCVI data (Febβ€“Jun only) ---
    field_spec = spectral[spectral["field_key"] == fk].copy()
    field_spec = field_spec[
        (field_spec["date"] >= fit_window_start) & (field_spec["date"] <= fit_window_end)
    ].sort_values("date")

    if "GCVI" not in field_spec.columns or field_spec.empty:
        continue

    # Use RAW dayofyear β€” matches fitted parameter scale
    field_doy = field_spec["date"].dt.dayofyear.values.astype(float)
    field_gcvi = field_spec["GCVI"].values

    # --- SG Smoothing (reproduce pipeline) ---
    fit_ts = field_spec.set_index("date")["GCVI"]
    daily_dates = pd.date_range(fit_window_start, fit_window_end, freq="1D")
    daily_interp = fit_ts.reindex(daily_dates).interpolate(method="linear", limit_direction="both")
    daily_interp = daily_interp.ffill().bfill()

    sg_win = min(7 * 3, len(daily_interp))
    if sg_win % 2 == 0:
        sg_win -= 1
    sg_win = max(sg_win, 5)
    smoothed_daily = savgol_filter(daily_interp.values, window_length=sg_win, polyorder=2)
    smooth_doy = daily_dates.dayofyear.values.astype(float)

    # --- Double logistic fitted curve (RAW DOY) ---
    params_row = dl_params[dl_params["field_key"] == fk].iloc[0]
    peak_row = valid_peaks[valid_peaks["field_key"] == fk].iloc[0]

    t_fit = np.linspace(fit_window_start.dayofyear, fit_window_end.dayofyear, 500)
    y_fit = double_logistic(
        t_fit,
        params_row["m1"], params_row["m2"], params_row["m3"],
        params_row["m4"], params_row["m5"], params_row["m6"], params_row["m7"],
    )

    # --- Traces ---
    # 1) Raw data (gray)
    fig3.add_trace(
        go.Scatter(
            x=field_doy, y=field_gcvi,
            mode="markers",
            marker=dict(size=5, color="#AAAAAA", opacity=0.6),
            name="Raw GCVI" if idx == 0 else None,
            showlegend=(idx == 0), legendgroup="raw",
        ),
        row=row, col=col,
    )

    # 2) SG smoothing (blue)
    fig3.add_trace(
        go.Scatter(
            x=smooth_doy, y=smoothed_daily,
            mode="lines",
            line=dict(color="#1F77B4", width=2),
            name="SG smoothing" if idx == 0 else None,
            showlegend=(idx == 0), legendgroup="sg",
        ),
        row=row, col=col,
    )

    # 3) Double logistic fit (red)
    fig3.add_trace(
        go.Scatter(
            x=t_fit, y=y_fit,
            mode="lines",
            line=dict(color="#E63946", width=2.5),
            name="Double logistic fit" if idx == 0 else None,
            showlegend=(idx == 0), legendgroup="dl",
        ),
        row=row, col=col,
    )

    # 4) Peak vertical line (black dashed, full height via shape)
    peak_doy_val = peak_row["peak_doy"]
    axis_suffix = str(idx + 1) if idx > 0 else ""
    fig3.add_shape(
        type="line",
        x0=peak_doy_val, x1=peak_doy_val, y0=0, y1=1,
        xref=f"x{axis_suffix}", yref=f"y{axis_suffix} domain",
        line=dict(color="black", width=1.5, dash="dash"),
    )

    # Annotation: RΒ², peak DOY
    xref_str = f"x{axis_suffix} domain"
    yref_str = f"y{axis_suffix} domain"
    fig3.add_annotation(
        text=f"RΒ² = {params_row['fit_r2']:.3f}<br>Peak DOY = {peak_row['peak_doy']:.0f}",
        x=0.95, y=0.95,
        xanchor="right", yanchor="top",
        xref=xref_str, yref=yref_str,
        showarrow=False, font=dict(size=11),
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#ccc", borderwidth=1,
    )

# Add a dummy trace for peak legend entry
fig3.add_trace(
    go.Scatter(
        x=[None], y=[None], mode="lines",
        line=dict(color="black", width=1.5, dash="dash"),
        name="Peak", showlegend=True, legendgroup="peak",
    )
)

# X-axis: month tick labels
month_ticks = pd.date_range("2025-02-01", "2025-07-01", freq="MS")
tick_vals = [d.dayofyear for d in month_ticks]
tick_labels = [d.strftime("%b") for d in month_ticks]

fig3.update_xaxes(
    tickvals=tick_vals, ticktext=tick_labels,
    title_text="Date (2025)", title_font=AXIS_FONT,
)
fig3.update_yaxes(title_text="GCVI", title_font=AXIS_FONT)
fig3.update_layout(
    title=dict(text="(a) Double Logistic Curve Fitting β€” Sample Fields", font=TITLE_FONT),
    font=dict(family=FONT_FAMILY, size=12),
    width=1000, height=700,
    legend=dict(x=0.5, y=-0.08, xanchor="center", orientation="h"),
)

fig3.show()
fig3.write_html(FIGURES_OUT / "fig3a_peak_fitting.html")
fig3.write_image(FIGURES_OUT / "fig3a_peak_fitting.png", scale=3)

In [ ]:
# Panel B: Peak DOY distribution
valid_peaks_only = peak_days[~peak_days["peak_anomalous"]]

fig3b = go.Figure()

fig3b.add_trace(
    go.Histogram(
        x=valid_peaks_only["peak_doy"],
        nbinsx=25,
        marker_color="#457B9D",
        marker_line=dict(color="white", width=0.8),
        opacity=0.85,
        name="Valid fields",
    )
)

# Add mean line
mean_doy = valid_peaks_only["peak_doy"].mean()
fig3b.add_vline(
    x=mean_doy, line_dash="dash", line_color="#E63946", line_width=2,
    annotation_text=f"Mean: DOY {mean_doy:.0f}",
    annotation_position="top right",
    annotation_font=dict(size=12, color="#E63946"),
)

# Add month labels on x-axis
fig3b.update_layout(
    title=dict(text="(b) Distribution of GCVI Peak Dates (n=228)", font=TITLE_FONT),
    xaxis=dict(
        title="Day of Year (2025)",
        title_font=AXIS_FONT,
        tickvals=[60, 91, 121, 152],
        ticktext=["Mar 1", "Apr 1", "May 1", "Jun 1"],
    ),
    yaxis=dict(title="Number of Fields", title_font=AXIS_FONT),
    font=dict(family=FONT_FAMILY, size=FONT_SIZE),
    width=700, height=400,
    showlegend=False,
)

# Stats annotation
stats_text = (
    f"Mean: {valid_peaks_only['peak_doy'].mean():.1f}<br>"
    f"Median: {valid_peaks_only['peak_doy'].median():.1f}<br>"
    f"Std: {valid_peaks_only['peak_doy'].std():.1f}<br>"
    f"Fit RΒ² mean: {valid_peaks_only['fit_r2'].mean():.3f}"
)
fig3b.add_annotation(
    text=stats_text, x=0.97, y=0.95,
    xref="paper", yref="paper",
    xanchor="right", yanchor="top",
    showarrow=False,
    font=dict(size=11),
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="#ccc", borderwidth=1,
)

fig3b.show()
fig3b.write_html(FIGURES_OUT / "fig3b_peak_distribution.html")
fig3b.write_image(FIGURES_OUT / "fig3b_peak_distribution.png", scale=3)

---
## Table 1 & Figure 5: Main Results (2TN Γ— 3TA Γ— 3M)

In [ ]:
# ============================================================
# Build Table 1 from temporal combination CSVs (correct R² values)
# Filter: only combos starting from September 2024 onwards
# ============================================================
STRATEGY_CSV_MAP = {
    "monthly": ("Peak-relative", "Monthly"),
    "biweekly": ("Peak-relative", "Biweekly"),
    "peak_stages": ("Peak-relative", "Periods"),
    "calendar_monthly": ("Calendar", "Monthly"),
    "calendar_biweekly": ("Calendar", "Biweekly"),
    "custom": ("Calendar", "Periods"),
}

# Periods to EXCLUDE per strategy (before September 2024)
EXCLUDE_PERIODS = {
    "monthly": {"mo_m9", "mo_m8"},           # Jul-Aug, Aug-Sep
    "biweekly": {"bw_m18", "bw_m17"},             # Jul, Aug
    "peak_stages": {"cp1"},                   # -300 to -121 days (~Jul 2024)
    "calendar_monthly": {"jul", "aug"},       # Jul, Aug
    "calendar_biweekly": {"jul_1", "jul_2", "aug_1", "aug_2"},
    "custom": {"cp1"},                        # Jul-Dec (includes Jul-Aug)
}

MODELS_TO_SHOW = ["RandomForest", "LightGBM", "XGBoost"]
METRICS_TO_SHOW = ["R2", "CCC", "RMSE", "RRMSE", "KGE"]

rows = []
best_combos = {}
best_models = {}  # best model name per strategy key

for csv_key, (norm, agg) in STRATEGY_CSV_MAP.items():
    if csv_key not in combo_results:
        print(f"  WARNING: {csv_key} not in combo_results, skipping")
        continue

    df = combo_results[csv_key].copy()
    n_before = len(df)

    # Filter: exclude combos whose start_period is before September
    exclude = EXCLUDE_PERIODS.get(csv_key, set())
    df = df[~df["start_period"].isin(exclude)]
    print(f"{norm} / {agg}: {n_before} → {len(df)} combos (excluded {n_before - len(df)} pre-Sep)")

    # Find best temporal combination (max R² across RF/LGBM/XGB)
    r2_cols = [f"{m}_R2_mean" for m in MODELS_TO_SHOW]
    df["best_R2_3models"] = df[r2_cols].max(axis=1)
    best_idx = df["best_R2_3models"].idxmax()
    best_row = df.loc[best_idx]

    combo_name = best_row["combo"]
    best_combos[csv_key] = combo_name
    best_model = MODELS_TO_SHOW[np.argmax([best_row[f"{m}_R2_mean"] for m in MODELS_TO_SHOW])]
    best_models[csv_key] = best_model
    print(f"  → best: {combo_name}  R²={best_row['best_R2_3models']:.4f} ({best_model})")

    for model in MODELS_TO_SHOW:
        row = {
            "Normalization": norm,
            "Aggregation": agg,
            "Model": model,
            "strategy_key": csv_key,
            "Best Combo": combo_name,
            "N Periods": int(best_row["n_periods"]),
        }
        for metric in METRICS_TO_SHOW:
            mean_col = f"{model}_{metric}_mean"
            std_col = f"{model}_{metric}_std"
            mean_val = best_row.get(mean_col, np.nan)
            std_val = best_row.get(std_col, np.nan)
            row[f"{metric}_mean"] = mean_val
            row[f"{metric}_std"] = std_val
            row[metric] = f"{mean_val:.3f} ± {std_val:.3f}"
        rows.append(row)

table1 = pd.DataFrame(rows)
print(f"\nTable 1: {len(table1)} rows (6 strategies × 3 models)")
print(f"Best models: {best_models}")

display_cols = ["Normalization", "Aggregation", "Model", "Best Combo"] + METRICS_TO_SHOW
table1[display_cols].sort_values(["Normalization", "Aggregation", "Model"])

In [ ]:
# Also extract Ensemble/Stacking for the best combo per strategy
extra_models = ["Ensemble", "Stacking"]
extra_rows = []

for csv_key, (norm, agg) in STRATEGY_CSV_MAP.items():
    if csv_key not in combo_results or csv_key not in best_combos:
        continue
    df = combo_results[csv_key]
    combo_name = best_combos[csv_key]
    best_row = df[df["combo"] == combo_name].iloc[0]

    for model in extra_models:
        r2_col = f"{model}_R2_mean"
        if r2_col not in df.columns:
            continue
        row = {
            "Normalization": norm,
            "Aggregation": agg,
            "Model": model,
            "strategy_key": csv_key,
            "Best Combo": combo_name,
            "N Periods": int(best_row["n_periods"]),
        }
        for metric in METRICS_TO_SHOW:
            mean_col = f"{model}_{metric}_mean"
            std_col = f"{model}_{metric}_std"
            mean_val = best_row.get(mean_col, np.nan)
            std_val = best_row.get(std_col, np.nan)
            row[f"{metric}_mean"] = mean_val
            row[f"{metric}_std"] = std_val
            row[metric] = f"{mean_val:.3f} Β± {std_val:.3f}"
        extra_rows.append(row)

table1_full = pd.concat([table1, pd.DataFrame(extra_rows)], ignore_index=True)

display_cols = ["Normalization", "Aggregation", "Model", "Best Combo"] + METRICS_TO_SHOW
table1_full[display_cols].sort_values(["Normalization", "Aggregation", "Model"])

In [ ]:
# Save Table 1 as CSV
table1_export = table1_full[display_cols].sort_values(["Normalization", "Aggregation", "Model"])
table1_export.to_csv(FIGURES_OUT / "table1_main_results.csv", index=False)
print("Saved table1_main_results.csv")

---
### Regenerate OOF Predictions for Best Combos
Re-run the full nested CV pipeline (with Boruta feature selection) for the best temporal
combination of each strategy. Results are cached to disk — only runs once.

In [ ]:
# Regenerate OOF predictions for ALL 6 strategies using the full pipeline
# Replicates exact methodology from notebook 07: raw feature filtering + Boruta + nested CV
# Results cached to models/temporal_combinations/oof_regenerated/ (only runs once)

import re, time, copy
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
from src.config import load_config
from src.modeling import nested_cv_pipeline
from sklearn.metrics import r2_score

base_config = load_config()

# Only run RF, LightGBM, XGBoost (disable ElasticNet/Stacking for speed)
cfg = copy.deepcopy(base_config)
cfg['modeling']['models'] = {
    'RandomForest': True,
    'LightGBM': True,
    'XGBoost': True,
    'ElasticNet': False,
    'CatBoost': False,
    'Stacking': False,
}

# Period names per strategy (matches generate_period_windows output)
PERIOD_NAMES = {
    "monthly": [f"mo_m{i}" for i in range(9, 0, -1)] + ["mo_peak"] + [f"mo_p{i}" for i in range(1, 4)],
    "biweekly": [f"bw_m{i}" for i in range(18, 0, -1)] + ["bw_peak"] + [f"bw_p{i}" for i in range(1, 6)],
    "peak_stages": ["cp1", "cp2", "cp3", "cp4", "cp5"],
    "calendar_monthly": ["jul", "aug", "sep", "oct", "nov", "dec", "jan", "feb", "mar", "apr", "may", "jun"],
    "calendar_biweekly": [f"{m}_{h}" for m in ["jul","aug","sep","oct","nov","dec","jan","feb","mar","apr","may","jun"] for h in ["1","2"]],
    "custom": ["cp1", "cp2", "cp3", "cp4", "cp5"],
}

# Raw feature filtering (same regex as notebook 07)
_P = (r'(?:bw|mo|cp[1-5]'
      r'|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec'
      r'|tillering|stem_elongation|booting|heading_anthesis|grain_filling|maturity)')
_derived_patterns = [
    rf'_(change|ratio|roc)_{_P}',
    r'_div_',
    r'_x_',
    r'_aridity_idx', r'_water_deficit', r'_heat_stress',
    r'^pheno_',
    r'_seasonal_cv$', r'_seasonal_range$', r'_seasonal_slope$',
]
_is_derived = lambda c: any(re.search(p, c) for p in _derived_patterns)

_exclude_cols = {'field_key', 'protein_pct', 'yield_bu_ac', 'county', 'state',
                 'centroid_lat', 'centroid_lon'}

# Cache directory
oof_cache_dir = TEMPORAL / "oof_regenerated"
oof_cache_dir.mkdir(exist_ok=True)

total_t0 = time.time()

for strategy_key in STRATEGY_CSV_MAP:
    cache_path = oof_cache_dir / f"oof_{strategy_key}.csv"

    # Load from cache if available
    if cache_path.exists():
        oof[strategy_key] = pd.read_csv(cache_path)
        n = len(oof[strategy_key])
        print(f"[cached] {strategy_key}: {n} rows from {cache_path.name}")
        continue

    print(f"\n{'='*60}")
    print(f"Regenerating OOF: {strategy_key}")
    print(f"{'='*60}")

    # 1. Load engineered features parquet
    parquet_path = NB_PROC / f"engineered_features_{strategy_key}.parquet"
    df_feat = pd.read_parquet(parquet_path)
    period_names = PERIOD_NAMES[strategy_key]

    # 2. Filter to raw features only (same as notebook 07)
    all_feat_cols = [c for c in df_feat.columns
                     if c not in _exclude_cols and df_feat[c].dtype in ('int64', 'float64')]
    raw_feat_cols = [c for c in all_feat_cols if not _is_derived(c)]
    static_cols = [c for c in raw_feat_cols
                   if not any(c.startswith(p + '_') for p in period_names)]

    # 3. Parse best combo -> list of contiguous periods
    best_combo = best_combos[strategy_key]
    start_p, end_p = best_combo.split("__to__")
    start_idx = period_names.index(start_p)
    end_idx = period_names.index(end_p)
    combo_periods = period_names[start_idx:end_idx + 1]

    # 4. Subset features to combo periods + static
    period_cols = [c for c in raw_feat_cols
                   if any(c.startswith(p + '_') for p in combo_periods)]
    feat_cols = period_cols + static_cols

    X_combo = df_feat[feat_cols].copy()
    y_vals = df_feat['protein_pct'].values
    groups_vals = df_feat['field_key'].values

    print(f"  Combo: {best_combo} ({len(combo_periods)} periods: {combo_periods})")
    print(f"  Features: {len(feat_cols)} ({len(period_cols)} temporal + {len(static_cols)} static)")

    # 5. Run nested CV pipeline (with Boruta feature selection)
    t0 = time.time()
    cv_result = nested_cv_pipeline(X_combo, y_vals, groups_vals, cfg, run_feature_selection=True)
    elapsed = time.time() - t0
    print(f"  Pipeline completed in {elapsed:.1f}s")

    # 6. Convert OOF predictions from long to wide format
    oof_raw = cv_result['oof_predictions']
    pred_cols = [c for c in oof_raw.columns if c.startswith('y_pred_')]
    oof_wide = oof_raw.groupby(['field_key', 'y_true'])[pred_cols].first().reset_index()

    # 7. Print R2 comparison (pooled vs mean-fold)
    for model in MODELS_TO_SHOW:
        col = f'y_pred_{model}'
        if col in oof_wide.columns:
            valid = oof_wide.dropna(subset=[col])
            r2_pooled = r2_score(valid['y_true'], valid[col])
            r2_folds = cv_result['summary'].get(model, {}).get('R2', [])
            r2_mean_fold = np.mean(r2_folds) if r2_folds else float('nan')
            print(f"  {model}: pooled R2={r2_pooled:.4f}, mean-fold R2={r2_mean_fold:.4f}")

    # 8. Cache and store
    oof_wide.to_csv(cache_path, index=False)
    oof[strategy_key] = oof_wide
    print(f"  Saved: {cache_path.name} ({len(oof_wide)} rows)")

total_elapsed = time.time() - total_t0
print(f"\nTotal time: {total_elapsed:.1f}s")
print(f"OOF strategies loaded: {list(oof.keys())}")

In [ ]:
# Figure 5: Performance Heatmap β€” Best model RΒ² per strategy
# Pivot: rows = Aggregation, columns = Normalization, value = best RΒ² across 3 models

# Get best RΒ² per strategy (across RF, LGB, XGB)
best_per_strategy = (
    table1.groupby(["Normalization", "Aggregation"])
    .agg({"R2_mean": "max", "CCC_mean": "max", "KGE_mean": "max"})
    .reset_index()
)

# Identify which model was best per strategy
best_model_per_strategy = (
    table1.loc[table1.groupby(["Normalization", "Aggregation"])["R2_mean"].idxmax()]
    [["Normalization", "Aggregation", "Model", "R2_mean", "Best Combo"]]
)

agg_order = ["Periods", "Monthly", "Biweekly"]
norm_order = ["Peak-relative", "Calendar"]

# Heatmap for each metric
for metric, label in [("R2_mean", "RΒ²"), ("CCC_mean", "CCC"), ("KGE_mean", "KGE")]:
    pivot = best_per_strategy.pivot(
        index="Aggregation", columns="Normalization", values=metric
    )
    pivot = pivot.reindex(index=agg_order, columns=norm_order)

    # Build text annotations (value + best model name + combo)
    text_matrix = []
    for agg in agg_order:
        row_text = []
        for norm in norm_order:
            val = pivot.loc[agg, norm] if agg in pivot.index and norm in pivot.columns else np.nan
            match = best_model_per_strategy[
                (best_model_per_strategy["Normalization"] == norm) &
                (best_model_per_strategy["Aggregation"] == agg)
            ]
            model_name = match["Model"].values[0] if len(match) > 0 else ""
            row_text.append(f"{val:.3f}<br>({model_name})" if not np.isnan(val) else "N/A")
        text_matrix.append(row_text)

    fig5_metric = go.Figure(
        data=go.Heatmap(
            z=pivot.values,
            x=pivot.columns.tolist(),
            y=pivot.index.tolist(),
            text=text_matrix,
            texttemplate="%{text}",
            textfont=dict(size=14, color="black"),
            colorscale="YlOrRd",
            reversescale=False,
            colorbar=dict(title=label),
            zmin=pivot.values[~np.isnan(pivot.values)].min() - 0.02 if not np.all(np.isnan(pivot.values)) else 0,
            zmax=pivot.values[~np.isnan(pivot.values)].max() + 0.02 if not np.all(np.isnan(pivot.values)) else 1,
        )
    )

    fig5_metric.update_layout(
        title=dict(text=f"Best {label} per Temporal Strategy (best combo, 3 models)", font=TITLE_FONT),
        xaxis=dict(title="Temporal Normalization", title_font=AXIS_FONT),
        yaxis=dict(title="Temporal Aggregation", title_font=AXIS_FONT, autorange="reversed"),
        font=dict(family=FONT_FAMILY, size=FONT_SIZE),
        width=600, height=400,
    )
    fig5_metric.show()
    fig5_metric.write_html(FIGURES_OUT / f"fig5_heatmap_{label.replace('Β²', '2')}.html")
    fig5_metric.write_image(FIGURES_OUT / f"fig5_heatmap_{label.replace('Β²', '2')}.png", scale=3)

In [ ]:
# Figure 5: Grouped bar chart — all 3 models per strategy (best temporal combo)

fig5_bar = go.Figure()

# Create strategy labels
table1["Strategy"] = table1["Normalization"] + " / " + table1["Aggregation"]
strategy_order = [
    "Peak-relative / Periods", "Peak-relative / Monthly", "Peak-relative / Biweekly",
    "Calendar / Periods", "Calendar / Monthly", "Calendar / Biweekly",
]

for model in MODELS_TO_SHOW:
    model_data = table1[table1["Model"] == model].copy()
    model_data = model_data.set_index("Strategy").reindex(strategy_order).reset_index()

    fig5_bar.add_trace(
        go.Bar(
            name=model,
            x=model_data["Strategy"],
            y=model_data["R2_mean"],
            error_y=dict(type="data", array=model_data["R2_std"], visible=True),
            marker_color=MODEL_COLORS[model],
            opacity=0.85,
        )
    )

# Dynamic y-axis range
max_r2 = table1["R2_mean"].max() + table1["R2_std"].max() + 0.05

fig5_bar.update_layout(
    title=dict(text="Model Performance Across Temporal Strategies (R²)", font=TITLE_FONT),
    xaxis=dict(title="Temporal Strategy", title_font=AXIS_FONT, tickangle=-30),
    yaxis=dict(title="R² (mean ± std, 5-fold CV)", title_font=AXIS_FONT, range=[0, max_r2]),
    barmode="group",
    font=dict(family=FONT_FAMILY, size=12),
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
    width=1000, height=500,
)

# Add separator between peak-relative and calendar
fig5_bar.add_vline(x=2.5, line_dash="dot", line_color="gray", line_width=1.5)
fig5_bar.add_annotation(
    x=1, y=max_r2 - 0.02, text="Peak-relative", showarrow=False,
    font=dict(size=12, color="#E63946"),
)
fig5_bar.add_annotation(
    x=4, y=max_r2 - 0.02, text="Calendar", showarrow=False,
    font=dict(size=12, color="#457B9D"),
)

fig5_bar.show()
fig5_bar.write_html(FIGURES_OUT / "fig5_bar_performance.html")
fig5_bar.write_image(FIGURES_OUT / "fig5_bar_performance.png", scale=3)

---
## Figure 6: Temporal Zoom-In (Performance per Period)

In [ ]:
# Load the monthly temporal combination results (peak-relative)
monthly_combos = combo_results.get("monthly", pd.DataFrame())

if not monthly_combos.empty:
    # Filter single-period combinations (n_periods == 1)
    single_period = monthly_combos[monthly_combos["n_periods"] == 1].copy()

    # Exclude pre-September periods
    single_period = single_period[~single_period["start_period"].isin(EXCLUDE_PERIODS["monthly"])]

    # Define period order (only Sep onwards: mo_m7 to mo_p3)
    period_order = [f"mo_m{i}" for i in range(7, 0, -1)] + ["mo_peak"] + [f"mo_p{i}" for i in range(1, 4)]
    period_labels = [f"M-{i}" for i in range(7, 0, -1)] + ["Peak"] + [f"M+{i}" for i in range(1, 4)]

    single_period["period_idx"] = single_period["start_period"].map(
        {p: i for i, p in enumerate(period_order)}
    )
    single_period = single_period.dropna(subset=["period_idx"]).sort_values("period_idx")

    label_map = dict(zip(period_order, period_labels))
    single_period["label"] = single_period["start_period"].map(label_map)

    fig6 = go.Figure()

    for model in MODELS_TO_SHOW:
        r2_col = f"{model}_R2_mean"
        r2_std_col = f"{model}_R2_std"
        if r2_col not in single_period.columns:
            continue

        fig6.add_trace(
            go.Scatter(
                x=single_period["label"],
                y=single_period[r2_col],
                mode="lines+markers",
                name=model,
                line=dict(color=MODEL_COLORS[model], width=2),
                marker=dict(size=8),
                error_y=dict(
                    type="data",
                    array=single_period[r2_std_col],
                    visible=True,
                    thickness=1,
                ),
            )
        )

    # Highlight Peak and M+1
    for lbl, clr in [("Peak", "#E63946"), ("M+1", "#FF9800")]:
        fig6.add_vline(
            x=lbl, line_dash="dot", line_color=clr, line_width=2, opacity=0.7,
        )

    fig6.update_layout(
        title=dict(
            text="Single-Period Predictive Power (Peak-Relative Monthly)",
            font=TITLE_FONT,
        ),
        xaxis=dict(
            title="Temporal Period (relative to GCVI peak)",
            title_font=AXIS_FONT,
            categoryorder="array",
            categoryarray=period_labels,
        ),
        yaxis=dict(title="R² (5-fold CV)", title_font=AXIS_FONT),
        font=dict(family=FONT_FAMILY, size=FONT_SIZE),
        width=1000, height=500,
        legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
        hovermode="x unified",
    )

    fig6.show()
    fig6.write_html(FIGURES_OUT / "fig6_temporal_zoom_monthly.html")
    fig6.write_image(FIGURES_OUT / "fig6_temporal_zoom_monthly.png", scale=3)
else:
    print("Monthly combo results not found")

In [ ]:
# Figure 6b: Cumulative performance (expanding window from peak) — all 3 models

if not monthly_combos.empty:
    # Filter out pre-September combos
    monthly_filtered = monthly_combos[~monthly_combos["start_period"].isin(EXCLUDE_PERIODS["monthly"])].copy()

    # Forward expansion: mo_peak → mo_p1, mo_peak → mo_p2, etc.
    forward = monthly_filtered[
        monthly_filtered["start_period"] == "mo_peak"
    ].sort_values("n_periods")

    if not forward.empty:
        fig6b = go.Figure()

        for model in MODELS_TO_SHOW:
            r2_col = f"{model}_R2_mean"
            if r2_col not in forward.columns:
                continue
            fig6b.add_trace(
                go.Scatter(
                    x=forward["combo"],
                    y=forward[r2_col],
                    mode="lines+markers",
                    name=model,
                    line=dict(color=MODEL_COLORS[model], width=2),
                    marker=dict(size=7),
                )
            )

        fig6b.update_layout(
            title=dict(
                text="Expanding Window from Peak (Forward)",
                font=TITLE_FONT,
            ),
            xaxis=dict(title="Period Range", title_font=AXIS_FONT, tickangle=-30),
            yaxis=dict(title="R²", title_font=AXIS_FONT),
            font=dict(family=FONT_FAMILY, size=12),
            width=800, height=450,
            legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
        )
        fig6b.show()
        fig6b.write_html(FIGURES_OUT / "fig6b_expanding_window.html")
        fig6b.write_image(FIGURES_OUT / "fig6b_expanding_window.png", scale=3)

---
## Figure 7: Predicted vs. Observed (Best Model)

In [ ]:
def plot_pred_vs_obs(oof_df, model_col, strategy_name, metrics_dict=None):
    """Create a publication-quality predicted vs observed scatter plot."""
    # Consolidate OOF predictions across folds
    col = f"y_pred_{model_col}"
    df = oof_df.dropna(subset=["y_true", col]).copy()
    
    y_true = df["y_true"].values
    y_pred = df[col].values
    
    # Compute metrics
    from sklearn.metrics import r2_score, mean_squared_error
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mbe = np.mean(y_pred - y_true)
    # CCC
    mean_t, mean_p = y_true.mean(), y_pred.mean()
    var_t, var_p = y_true.var(), y_pred.var()
    sd_t, sd_p = y_true.std(), y_pred.std()
    rho = np.corrcoef(y_true, y_pred)[0, 1]
    ccc_val = 2 * rho * sd_t * sd_p / (var_t + var_p + (mean_t - mean_p)**2 + 1e-10)
    
    # Plot
    lims = [
        min(y_true.min(), y_pred.min()) - 0.5,
        max(y_true.max(), y_pred.max()) + 0.5,
    ]
    
    fig = go.Figure()
    
    # Β±RMSE band
    x_line = np.linspace(lims[0], lims[1], 100)
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([x_line, x_line[::-1]]),
            y=np.concatenate([x_line + rmse, (x_line - rmse)[::-1]]),
            fill="toself",
            fillcolor="rgba(69, 123, 157, 0.15)",
            line=dict(color="rgba(69, 123, 157, 0.3)", width=1, dash="dash"),
            name=f"Β±RMSE ({rmse:.2f}%)",
            hoverinfo="skip",
        )
    )
    
    # 1:1 line
    fig.add_trace(
        go.Scatter(
            x=lims, y=lims,
            mode="lines",
            line=dict(color="black", width=1.5),
            name="1:1 line",
            hoverinfo="skip",
        )
    )
    
    # Data points
    fig.add_trace(
        go.Scatter(
            x=y_true, y=y_pred,
            mode="markers",
            marker=dict(
                size=7,
                color="#457B9D",
                opacity=0.7,
                line=dict(color="white", width=0.5),
            ),
            name="Observations",
            text=df["field_key"],
            hovertemplate="Field: %{text}<br>Observed: %{x:.2f}%<br>Predicted: %{y:.2f}%",
        )
    )
    
    # Metrics annotation
    metrics_text = (
        f"RΒ² = {r2:.3f}<br>"
        f"CCC = {ccc_val:.3f}<br>"
        f"RMSE = {rmse:.3f}%<br>"
        f"MBE = {mbe:.3f}%<br>"
        f"n = {len(y_true)}"
    )
    fig.add_annotation(
        text=metrics_text,
        x=0.05, y=0.95,
        xref="paper", yref="paper",
        xanchor="left", yanchor="top",
        showarrow=False,
        font=dict(size=12, family="monospace"),
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="#ccc", borderwidth=1,
    )
    
    fig.update_layout(
        title=dict(
            text=f"Predicted vs Observed GPC {model_col} ({strategy_name})",
            font=TITLE_FONT,
        ),
        xaxis=dict(
            title="Observed Protein (%)",
            title_font=AXIS_FONT,
            range=lims,
            scaleanchor="y",
            constrain="domain",
        ),
        yaxis=dict(
            title="Predicted Protein (%)",
            title_font=AXIS_FONT,
            range=lims,
        ),
        font=dict(family=FONT_FAMILY, size=FONT_SIZE),
        width=600, height=600,
        legend=dict(x=0.55, y=0.05, bgcolor="rgba(255,255,255,0.8)"),
    )
    
    return fig

print("plot_pred_vs_obs function defined.")

In [ ]:
# Plot best model for peak-relative monthly (the overall best strategy)
best_model_monthly = best_models.get("monthly", "RandomForest")
fig7_monthly = plot_pred_vs_obs(
    oof["monthly"], best_model_monthly, "Peak-relative Monthly"
)
fig7_monthly.show()
fig7_monthly.write_html(FIGURES_OUT / f"fig7_pred_vs_obs_{best_model_monthly.lower()}_monthly.html")
fig7_monthly.write_image(FIGURES_OUT / f"fig7_pred_vs_obs_{best_model_monthly.lower()}_monthly.png", scale=3)

In [ ]:
# Multi-panel: best model per strategy (all 6 strategies)
strategy_labels = {
    "monthly": "Peak-relative / Monthly",
    "biweekly": "Peak-relative / Biweekly",
    "peak_stages": "Peak-relative / Periods",
    "calendar_monthly": "Calendar / Monthly",
    "calendar_biweekly": "Calendar / Biweekly",
    "custom": "Calendar / Periods",
}

for strategy_key, label in strategy_labels.items():
    model = best_models.get(strategy_key)
    if model and strategy_key in oof:
        fig = plot_pred_vs_obs(oof[strategy_key], model, label)
        fig.show()
        fig.write_image(
            FIGURES_OUT / f"fig7_pred_vs_obs_{model.lower()}_{strategy_key}.png",
            scale=3,
        )

---
## Figure 7b: Confusion Matrices (3-class protein classification)

In [ ]:
# Confusion matrices: classify protein into 3 classes and evaluate
from sklearn.metrics import confusion_matrix

LOW_THRESH = 11.4
HIGH_THRESH = 12.8

def classify_3class(vals):
    classes = np.empty(len(vals), dtype=object)
    classes[vals < LOW_THRESH] = 'Low (<11.4)'
    classes[(vals >= LOW_THRESH) & (vals <= HIGH_THRESH)] = 'Mid (11.4-12.8)'
    classes[vals > HIGH_THRESH] = 'High (>12.8)'
    return classes

class_labels = ['Low (<11.4)', 'Mid (11.4-12.8)', 'High (>12.8)']

# Use OOF predictions for best strategy (monthly) — average per field
oof_monthly = oof['monthly'].copy()
oof_avg = oof_monthly.groupby('field_key').agg({
    'y_true': 'first',
    'y_pred_RandomForest': 'mean',
    'y_pred_LightGBM': 'mean',
    'y_pred_XGBoost': 'mean',
}).reset_index()

best_combo_name = best_combos.get('monthly', 'mo_p1__to__mo_p1')

fig_cm = make_subplots(
    rows=1, cols=3,
    subplot_titles=['RandomForest', 'LightGBM', 'XGBoost'],
    horizontal_spacing=0.12,
)

y_true_class = classify_3class(oof_avg['y_true'].values)

for col_idx, model in enumerate(MODELS_TO_SHOW, start=1):
    y_pred_class = classify_3class(oof_avg[f'y_pred_{model}'].values)
    cm = confusion_matrix(y_true_class, y_pred_class, labels=class_labels)
    accuracy = np.trace(cm) / cm.sum()

    fig_cm.add_trace(
        go.Heatmap(
            z=cm[::-1],
            x=class_labels,
            y=class_labels[::-1],
            colorscale='Blues',
            showscale=(col_idx == 3),
            text=cm[::-1],
            texttemplate='%{text}',
            textfont=dict(size=14),
        ),
        row=1, col=col_idx,
    )

    fig_cm.layout.annotations[col_idx - 1].update(
        text=f'{model}<br>Accuracy={accuracy:.3f}'
    )
    fig_cm.update_xaxes(title_text='Actual values', row=1, col=col_idx)
    fig_cm.update_yaxes(title_text='Predicted values' if col_idx == 1 else '', row=1, col=col_idx)

fig_cm.update_layout(
    title=dict(
        text=f'Confusion Matrices (3 classes) — {best_combo_name}',
        font=TITLE_FONT,
    ),
    font=dict(family=FONT_FAMILY, size=12),
    height=500,
    width=1200,
)

fig_cm.show()
fig_cm.write_html(FIGURES_OUT / 'fig7b_confusion_matrices.html')
fig_cm.write_image(FIGURES_OUT / 'fig7b_confusion_matrices.png', scale=3)

# Print accuracies
for model in MODELS_TO_SHOW:
    y_pred_class = classify_3class(oof_avg[f'y_pred_{model}'].values)
    cm = confusion_matrix(y_true_class, y_pred_class, labels=class_labels)
    acc = np.trace(cm) / cm.sum()
    print(f'{model}: accuracy={acc:.3f}')

---
## Figure 8: SHAP Analysis

In [ ]:
# Load SHAP values for the best temporal combination
shap_dir = TEMPORAL / "selected_monthly_analysis" / "shap"

# Find available SHAP files
shap_files = sorted(shap_dir.glob("*_shap_values.csv")) if shap_dir.exists() else []
print(f"SHAP files found: {len(shap_files)}")
for f in shap_files:
    print(f"  {f.name}")

In [ ]:
def plot_shap_beeswarm_plotly(shap_csv_path, title="SHAP Feature Importance", top_n=20):
    """Create a SHAP beeswarm-style plot using Plotly."""
    shap_df = pd.read_csv(shap_csv_path)
    
    # Mean absolute SHAP per feature
    mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)
    top_features = mean_abs_shap.head(top_n).index.tolist()
    
    fig = go.Figure()
    
    # Horizontal bar chart of mean |SHAP|
    fig.add_trace(
        go.Bar(
            y=top_features[::-1],
            x=mean_abs_shap[top_features].values[::-1],
            orientation="h",
            marker=dict(
                color=mean_abs_shap[top_features].values[::-1],
                colorscale="YlOrRd",
                colorbar=dict(title="Mean |SHAP|"),
            ),
        )
    )
    
    fig.update_layout(
        title=dict(text=title, font=TITLE_FONT),
        xaxis=dict(title="Mean |SHAP value|", title_font=AXIS_FONT),
        yaxis=dict(title="", tickfont=dict(size=10)),
        font=dict(family=FONT_FAMILY, size=12),
        width=800,
        height=max(400, top_n * 25),
        showlegend=False,
    )
    
    return fig, mean_abs_shap


def plot_shap_beeswarm_detailed(shap_csv_path, feature_csv_path=None, title="SHAP", top_n=15):
    """Create a detailed beeswarm-like plot showing SHAP value distribution per feature."""
    shap_df = pd.read_csv(shap_csv_path)
    
    # Mean absolute SHAP
    mean_abs = shap_df.abs().mean().sort_values(ascending=False)
    top_feats = mean_abs.head(top_n).index.tolist()
    
    fig = go.Figure()
    
    for i, feat in enumerate(top_feats[::-1]):
        vals = shap_df[feat].values
        jitter = np.random.normal(0, 0.12, len(vals))
        
        fig.add_trace(
            go.Scatter(
                x=vals,
                y=[i] * len(vals) + jitter,
                mode="markers",
                marker=dict(
                    size=4,
                    color=vals,
                    colorscale="RdBu_r",
                    opacity=0.6,
                    cmid=0,
                    colorbar=dict(title="SHAP value") if i == 0 else None,
                    showscale=(i == 0),
                ),
                name=feat,
                showlegend=False,
                hovertemplate=f"{feat}<br>SHAP: %{{x:.4f}}",
            )
        )
    
    fig.update_layout(
        title=dict(text=title, font=TITLE_FONT),
        xaxis=dict(title="SHAP value (impact on prediction)", title_font=AXIS_FONT),
        yaxis=dict(
            tickvals=list(range(top_n)),
            ticktext=top_feats[::-1],
            tickfont=dict(size=10),
        ),
        font=dict(family=FONT_FAMILY, size=12),
        width=900,
        height=max(500, top_n * 35),
        showlegend=False,
    )
    fig.add_vline(x=0, line_dash="dash", line_color="gray", line_width=1)
    
    return fig

print("SHAP plotting functions defined.")

In [ ]:
# Plot SHAP for the best combination + best model (mo_p1, RandomForest)
best_model_monthly = best_models.get("monthly", "RandomForest")
shap_priorities = [
    f"mo_p1__to__mo_p1_{best_model_monthly}_shap_values.csv",
    f"mo_p1__to__mo_p2_{best_model_monthly}_shap_values.csv",
    f"mo_peak__to__mo_p3_{best_model_monthly}_shap_values.csv",
]

for shap_name in shap_priorities:
    shap_path = shap_dir / shap_name
    if shap_path.exists():
        model_name = shap_name.split("_shap")[0]

        # Bar chart version
        fig8a, importance = plot_shap_beeswarm_plotly(
            shap_path,
            title=f"SHAP Feature Importance — {model_name}",
            top_n=20,
        )
        fig8a.show()
        fig8a.write_image(FIGURES_OUT / f"fig8_shap_bar_{model_name}.png", scale=3)

        # Beeswarm version
        fig8b = plot_shap_beeswarm_detailed(
            shap_path,
            title=f"SHAP Beeswarm — {model_name}",
            top_n=15,
        )
        fig8b.show()
        fig8b.write_html(FIGURES_OUT / f"fig8_shap_beeswarm_{model_name}.html")
        fig8b.write_image(FIGURES_OUT / f"fig8_shap_beeswarm_{model_name}.png", scale=3)

        print(f"\nTop 10 features ({model_name}):")
        print(importance.head(10).to_string())
        break

---
## Figure 4: Temporal Strategies Schematic

In [ ]:
# Create a schematic showing all 6 temporal strategies
# Periods from September 2024 onwards (monthly/biweekly), full periods shown for context

import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig4 = make_subplots(
    rows=6, cols=1,
    subplot_titles=[
        "(a) Peak-relative: Periods (5 windows)",
        "(b) Peak-relative: Monthly (11 windows)",
        "(c) Peak-relative: Biweekly (21 windows)",
        "(d) Calendar: Periods (5 windows)",
        "(e) Calendar: Monthly (10 windows)",
        "(f) Calendar: Biweekly (20 windows)",
    ],
    vertical_spacing=0.04,
    shared_xaxes=True,
)

colors_periods = ["#264653", "#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"]

# --- Peak-relative strategies ---
# Periods (all 5 including CP1)
peak_periods = [
    ("CP1", -238, -121), ("CP2", -120, -60), ("CP3", -59, -14),
    ("CP4", -15, 15), ("CP5", 16, 36),
]
for i, (name, start, end) in enumerate(peak_periods):
    fig4.add_shape(
        type="rect", x0=start, x1=end, y0=0, y1=1,
        fillcolor=colors_periods[i], opacity=0.7,
        line=dict(color="white", width=1),
        row=1, col=1,
    )
    fig4.add_annotation(
        x=(start + end) / 2, y=0.5, text=name,
        showarrow=False, font=dict(size=9, color="white"),
        xref="x", yref="y",
        row=1, col=1,
    )

# Monthly (peak-relative) — exclude mo_m9 and mo_m8, start from mo_m7 (i=7)
for i in range(7, 0, -1):
    start = -15 - 1 - (i-1)*30 - 30 + 1
    end = -15 - 1 - (i-1)*30
    color = "#457B9D" if i > 1 else "#E63946"
    fig4.add_shape(
        type="rect", x0=start, x1=end, y0=0, y1=1,
        fillcolor=color, opacity=0.5,
        line=dict(color="white", width=0.5),
        row=2, col=1,
    )

# Peak month
fig4.add_shape(
    type="rect", x0=-15, x1=15, y0=0, y1=1,
    fillcolor="#E63946", opacity=0.8,
    line=dict(color="white", width=1),
    row=2, col=1,
)
fig4.add_annotation(
    x=0, y=0.5, text="Peak",
    showarrow=False, font=dict(size=9, color="white"),
    row=2, col=1,
)

# Post-peak months
for i in range(1, 4):
    start = 15 + 1 + (i-1)*30
    end = start + 30 - 1
    fig4.add_shape(
        type="rect", x0=start, x1=end, y0=0, y1=1,
        fillcolor="#E9C46A", opacity=0.6,
        line=dict(color="white", width=0.5),
        row=2, col=1,
    )

# Biweekly (peak-relative) — exclude bw_m18, bw_m17, bw_m16; start from bw_m15 (i=15)
for i in range(15, 0, -1):
    start = -7 - i*15
    end = start + 14
    fig4.add_shape(
        type="rect", x0=start, x1=end, y0=0, y1=1,
        fillcolor="#457B9D", opacity=0.4,
        line=dict(color="white", width=0.3),
        row=3, col=1,
    )
# Peak biweek
fig4.add_shape(
    type="rect", x0=-7, x1=7, y0=0, y1=1,
    fillcolor="#E63946", opacity=0.8,
    line=dict(color="white", width=1),
    row=3, col=1,
)
for i in range(1, 6):
    start = 7 + 1 + (i-1)*15
    end = start + 14
    fig4.add_shape(
        type="rect", x0=start, x1=end, y0=0, y1=1,
        fillcolor="#E9C46A", opacity=0.5,
        line=dict(color="white", width=0.3),
        row=3, col=1,
    )

# --- Calendar strategies (convert to DOY-relative for display) ---
# Peak DOY ~ 117 (mean), so day 0 = DOY 117+365 = 482
PEAK_DOY = 117

# Calendar periods — all 5 including CP1
cal_periods = [
    ("CP1", 244, 365),   # Sep 1 - Dec 31
    ("CP2", 366, 424),   # Jan 1 - Feb 28
    ("CP3", 425, 470),   # Mar 1 - Apr 15
    ("CP4", 471, 500),   # Apr 16 - May 15
    ("CP5", 501, 546),   # May 16 - Jun 30
]
for i, (name, doy_s, doy_e) in enumerate(cal_periods):
    rel_s = doy_s - (PEAK_DOY + 365)
    rel_e = doy_e - (PEAK_DOY + 365)
    fig4.add_shape(
        type="rect", x0=rel_s, x1=rel_e, y0=0, y1=1,
        fillcolor=colors_periods[i], opacity=0.7,
        line=dict(color="white", width=1),
        row=4, col=1,
    )
    fig4.add_annotation(
        x=(rel_s + rel_e) / 2, y=0.5, text=name,
        showarrow=False, font=dict(size=9, color="white"),
        row=4, col=1,
    )

# Calendar monthly — exclude Jul, Aug; show Sep-Jun (10 months)
month_starts = [244, 274, 305, 335, 366, 397, 425, 456, 486, 517]
month_ends =   [273, 304, 334, 365, 396, 424, 455, 485, 516, 546]
month_names =  ["Sep", "Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun"]

for i, (ms, me, mn) in enumerate(zip(month_starts, month_ends, month_names)):
    rel_s = ms - (PEAK_DOY + 365)
    rel_e = me - (PEAK_DOY + 365)
    color = "#457B9D" if i < 7 else "#E9C46A"
    if mn == "Apr":
        color = "#E63946"  # Peak month
    fig4.add_shape(
        type="rect", x0=rel_s, x1=rel_e, y0=0, y1=1,
        fillcolor=color, opacity=0.5,
        line=dict(color="white", width=0.5),
        row=5, col=1,
    )

# Calendar biweekly — exclude Jul_1, Jul_2, Aug_1, Aug_2; show Sep_1 onwards (20)
for i in range(20):
    m_idx = i // 2
    half = i % 2
    if half == 0:
        s = month_starts[m_idx]
        e = month_starts[m_idx] + 14
    else:
        s = month_starts[m_idx] + 15
        e = month_ends[m_idx]
    rel_s = s - (PEAK_DOY + 365)
    rel_e = e - (PEAK_DOY + 365)
    color = "#457B9D" if m_idx < 7 else "#E9C46A"
    fig4.add_shape(
        type="rect", x0=rel_s, x1=rel_e, y0=0, y1=1,
        fillcolor=color, opacity=0.4,
        line=dict(color="white", width=0.3),
        row=6, col=1,
    )

# Add peak line across all subplots
for row_idx in range(1, 7):
    fig4.add_vline(
        x=0, line_dash="dash", line_color="red", line_width=1.5,
        row=row_idx, col=1,
    )

fig4.update_xaxes(
    range=[-245, 100],
    title_text="Days relative to GCVI peak",
    title_font=AXIS_FONT,
    row=6, col=1,
)
fig4.update_yaxes(visible=False)

fig4.update_layout(
    title=dict(text="Temporal Aggregation Strategies (2 Normalizations \u00d7 3 Resolutions)", font=TITLE_FONT),
    font=dict(family=FONT_FAMILY, size=11),
    width=1100, height=800,
    showlegend=False,
)

fig4.show()
fig4.write_html(FIGURES_OUT / "fig4_temporal_strategies.html")
fig4.write_image(FIGURES_OUT / "fig4_temporal_strategies.png", scale=3)

---
## Supplementary: Protein Distribution

In [ ]:
# Protein distribution histogram (using all 228 valid fields)
fig_s1 = go.Figure()
fig_s1.add_trace(
    go.Histogram(
        x=fields_228["protein_pct"],
        nbinsx=20,
        marker_color="#457B9D",
        marker_line=dict(color="white", width=0.8),
        opacity=0.85,
    )
)

mean_prot = fields_228["protein_pct"].mean()
std_prot = fields_228["protein_pct"].std()

fig_s1.add_vline(
    x=mean_prot, line_dash="dash", line_color="#E63946", line_width=2,
    annotation_text=f"Mean: {mean_prot:.1f}%",
    annotation_position="top right",
)

fig_s1.update_layout(
    title=dict(text=f"Distribution of Grain Protein Content (n={len(fields_228)})", font=TITLE_FONT),
    xaxis=dict(title="Grain Protein Content (%)", title_font=AXIS_FONT),
    yaxis=dict(title="Number of Fields", title_font=AXIS_FONT),
    font=dict(family=FONT_FAMILY, size=FONT_SIZE),
    width=700, height=400,
)

fig_s1.add_annotation(
    text=(
        f"Mean: {mean_prot:.2f}%<br>"
        f"Std: {std_prot:.2f}%<br>"
        f"Range: {fields_228['protein_pct'].min():.1f} - {fields_228['protein_pct'].max():.1f}%"
    ),
    x=0.97, y=0.95,
    xref="paper", yref="paper",
    xanchor="right", yanchor="top",
    showarrow=False,
    font=dict(size=11),
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="#ccc", borderwidth=1,
)

fig_s1.show()
fig_s1.write_image(FIGURES_OUT / "figS1_protein_distribution.png", scale=3)

---
## Supplementary: Fit Quality Distribution

In [ ]:
# RΒ² distribution of double logistic fits
fig_s2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["(a) Fit RΒ² Distribution", "(b) Peak DOY vs Fit RΒ²"],
)

fig_s2.add_trace(
    go.Histogram(
        x=valid_peaks_only["fit_r2"],
        nbinsx=25,
        marker_color="#2a9d8f",
        marker_line=dict(color="white", width=0.5),
        name="Fit RΒ²",
    ),
    row=1, col=1,
)

fig_s2.add_trace(
    go.Scatter(
        x=valid_peaks_only["peak_doy"],
        y=valid_peaks_only["fit_r2"],
        mode="markers",
        marker=dict(size=5, color="#457B9D", opacity=0.6),
        name="Fields",
    ),
    row=1, col=2,
)

fig_s2.update_xaxes(title_text="Fit RΒ²", row=1, col=1)
fig_s2.update_xaxes(title_text="Peak DOY", row=1, col=2)
fig_s2.update_yaxes(title_text="Count", row=1, col=1)
fig_s2.update_yaxes(title_text="Fit RΒ²", row=1, col=2)

fig_s2.update_layout(
    title=dict(text="Double Logistic Fit Quality (n=228)", font=TITLE_FONT),
    font=dict(family=FONT_FAMILY, size=12),
    width=1000, height=400,
    showlegend=False,
)

fig_s2.show()
fig_s2.write_image(FIGURES_OUT / "figS2_fit_quality.png", scale=3)

---
## Table 2: Top Selected Features

In [ ]:
# Load feature selection data from the selected monthly analysis
feat_dir = TEMPORAL / "selected_monthly_analysis" / "features"

if feat_dir.exists():
    feat_files = sorted(feat_dir.glob("*_fold_features.csv"))
    print(f"Feature files: {len(feat_files)}")
    
    all_features = []
    for f in feat_files:
        df = pd.read_csv(f)
        all_features.append(df)
    
    if all_features:
        feat_df = pd.concat(all_features, ignore_index=True)
        
        # Count frequency across folds and combinations
        feat_freq = (
            feat_df.groupby("Feature")
            .agg(
                n_appearances=("Fold", "count"),
                combinations=("Combination", "nunique"),
            )
            .sort_values("n_appearances", ascending=False)
        )
        
        # Categorize features
        def categorize_feature(name):
            name_lower = name.lower()
            if "soil" in name_lower:
                return "Soil"
            elif "elev" in name_lower or "slope" in name_lower or "aspect" in name_lower:
                return "Topographic"
            elif any(m in name_lower for m in ["t2m", "precip", "pet", "vpd", "gdd", "dewpoint", "ssrd"]):
                return "Meteorological"
            elif "_x_" in name:
                return "Interaction"
            elif any(d in name for d in ["_change_", "_ratio_", "_roc_"]):
                return "Temporal Derivative"
            elif "_div_" in name:
                return "Band Ratio"
            else:
                return "Spectral/VI"
        
        feat_freq["Category"] = [categorize_feature(f) for f in feat_freq.index]
        
        # Table 2: Top 20 features
        table2 = feat_freq.head(20).reset_index()
        table2.columns = ["Feature", "Appearances (across folds)", "Combinations", "Category"]
        table2 = table2[["Feature", "Category", "Appearances (across folds)", "Combinations"]]
        
        table2.to_csv(FIGURES_OUT / "table2_top_features.csv", index=False)
        display(table2)
        
        # Category summary
        print("\nFeature category distribution (top 20):")
        print(table2["Category"].value_counts())
else:
    print("Feature selection directory not found")

---
## Summary: All Generated Files

In [ ]:
print("=" * 60)
print("GENERATED FILES")
print("=" * 60)

for f in sorted(FIGURES_OUT.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:50s} {size_kb:8.1f} KB")

print(f"\nTotal files: {len(list(FIGURES_OUT.glob('*')))}")
print(f"Output directory: {FIGURES_OUT}")